Este colab irá servir para extrair caracteristicas quantitativas sobre o dataset como:
- qtde de imagens
- qtde de imagens por classe
- qtde de pixels por classe
- proporção de pixels de fundo x daninha por classe
- classe com mais ocorrencia 

In [75]:
dataset_path = '/media/guatambu/hdd/wesley/daninhas_multiclasse/'
classes = ['DATASET_CARURU', 'DATASET_GRAMINEA_PORTE_ALTO', 'DATASET_GRAMINEA_PORTE_BAIXO', 'DATASET_MAMONA', 'DATASET_OUTRAS_FOLHAS_LARGAS', 'DATASET_TREPADEIRA']
#classes = ['DATASET_CARURU']

import os
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from PIL import Image
from tqdm import tqdm

# Lista de classes, cores únicas e contagens
# Para cada label, obtem os pixels únicos e suas contagens. Se ainda não existir aquele pixel unico, adiciona ele na lista de cores únicas e inicia a contagem. Se já existir, apenas atualiza a contagem.

classes_info = []

for cls in classes:
    print(f"Verificando a classe: {cls}")
    
    classes_info.append({cls: {}})
    
    labels_dir = os.path.join(dataset_path, cls, 'labels')

    images = os.listdir(labels_dir)
    
    for mask_file in tqdm(images, desc=f"Processando {cls}"):
        mask_path = os.path.join(labels_dir, mask_file)
        mask_np = np.array(Image.open(mask_path).convert('RGB'))
        pixels = mask_np.reshape(-1, 3)
        unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

        if sum(counts) != 65536:
            print(f"AVISO: O número total de pixels é {sum(counts)}, mas deveria ser 65536 (256x256). Verifique a imagem {cls}/{mask_file}!")
        for color, count in zip(unique_colors, counts):
            if np.array_equal(color, [0, 0, 0]):
                nome_cor = 'preto'
            elif np.array_equal(color, [255, 0, 0]):
                nome_cor = 'vermelho'
            elif np.array_equal(color, [0, 255, 0]):
                nome_cor = 'verde'
            elif np.array_equal(color, [0, 0, 255]):
                nome_cor = 'azul'
            elif np.array_equal(color, [255, 255, 0]):
                nome_cor = 'amarelo'
            elif np.array_equal(color, [255, 255, 255]):
                nome_cor = 'branco'
            else:
                nome_cor = f"RGB({color[0]},{color[1]},{color[2]})"
                print(f"AVISO: Cor desconhecida encontrada na classe {cls}: {nome_cor} (verifique a imagem {cls}/{mask_file})")
            
            if nome_cor not in classes_info[-1][cls]:
                classes_info[-1][cls][nome_cor] = count
            else:
                classes_info[-1][cls][nome_cor] += count
                
classes_info
        

Verificando a classe: DATASET_CARURU


Processando DATASET_CARURU: 100%|██████████| 97/97 [00:03<00:00, 26.71it/s]


Verificando a classe: DATASET_GRAMINEA_PORTE_ALTO


Processando DATASET_GRAMINEA_PORTE_ALTO: 100%|██████████| 2726/2726 [01:44<00:00, 26.21it/s]


Verificando a classe: DATASET_GRAMINEA_PORTE_BAIXO


...

Verificando a classe: DATASET_OUTRAS_FOLHAS_LARGAS


Processando DATASET_OUTRAS_FOLHAS_LARGAS: 100%|██████████| 2999/2999 [01:53<00:00, 26.32it/s]


Verificando a classe: DATASET_TREPADEIRA


Processando DATASET_TREPADEIRA: 100%|██████████| 2320/2320 [01:27<00:00, 26.41it/s]


[{'DATASET_CARURU': {'preto': np.int64(3147115),
   'vermelho': np.int64(751345),
   'branco': np.int64(2458532)}},
 {'DATASET_GRAMINEA_PORTE_ALTO': {'preto': np.int64(88368533),
   'branco': np.int64(85252832),
   'vermelho': np.int64(5029771)}},
 {'DATASET_GRAMINEA_PORTE_BAIXO': {'preto': np.int64(52153700),
   'vermelho': np.int64(2769671),
   'branco': np.int64(59568021)}},
 {'DATASET_MAMONA': {'preto': np.int64(43306444),
   'branco': np.int64(27419977),
   'vermelho': np.int64(1559787)}},
 {'DATASET_OUTRAS_FOLHAS_LARGAS': {'preto': np.int64(82433787),
   'branco': np.int64(110255869),
   'vermelho': np.int64(3852808)}},
 {'DATASET_TREPADEIRA': {'preto': np.int64(71302650),
   'vermelho': np.int64(8325171),
   'branco': np.int64(72415699)}}]

In [103]:
for classes_dict in classes_info:
    for cls, colors in classes_dict.items():
        print(f"\nClasse: {cls}\nTotal de pixels: {sum(colors.values())}")
        for color_name, count in colors.items():
            if color_name == 'preto':
                label = 'fundo'
            elif color_name == 'branco':
                label = 'desconhecido'
            else:
                label = 'daninha'
            
            if label == 'desconhecido':
                print(f"Label: {label} ({color_name})\nQuantidade de pixels: {count}\nProporção do label x total de pixels: {(count / sum(colors.values())) * 100:.2f}%\nProporção do label x pixels de fundo e daninha: N/A")
            else:
                print(f"Label: {label} ({color_name})\nQuantidade de pixels: {count}\nProporção do label x total de pixels: {(count / sum(colors.values())) * 100:.2f}%\nProporção do label x pixels de fundo e daninha: {((count / (sum(colors.values()) - colors.get('branco', 0))) * 100):.2f}%")
            print("-" * 50)


Classe: DATASET_CARURU
Total de pixels: 6356992
Label: fundo (preto)
Quantidade de pixels: 3147115
Proporção do label x total de pixels: 49.51%
Proporção do label x pixels de fundo e daninha: 80.73%
--------------------------------------------------
Label: daninha (vermelho)
Quantidade de pixels: 751345
Proporção do label x total de pixels: 11.82%
Proporção do label x pixels de fundo e daninha: 19.27%
--------------------------------------------------
Label: desconhecido (branco)
Quantidade de pixels: 2458532
Proporção do label x total de pixels: 38.67%
Proporção do label x pixels de fundo e daninha: N/A
--------------------------------------------------

Classe: DATASET_GRAMINEA_PORTE_ALTO
Total de pixels: 178651136
Label: fundo (preto)
Quantidade de pixels: 88368533
Proporção do label x total de pixels: 49.46%
Proporção do label x pixels de fundo e daninha: 94.61%
--------------------------------------------------
Label: desconhecido (branco)
Quantidade de pixels: 85252832
Proporção